# 02. 형태학 특징 추출기 — 정의와 검증

**설계 원칙**

> PPG는 절대 스케일을 잃고, 타이밍과 정규화된 형태를 보존한다.

PPG의 진폭은 접촉압·피부색·기기·측정부위에 좌우되므로 **절대 진폭은 특징으로 쓰지 않는다.**
모든 특징을 시간·비율·정규화 진폭으로만 정의한다.

| 계열 | 특징 |
|---|---|
| 타이밍 | CT(상승시간), LVET(notch까지), **CT/LVET**, dT, 폭(25/50/75%) |
| 비율 | RI(반사지수), IPA(면적비), notch 상대높이 |
| 1차 미분 | 최대 상승기울기(정규화), 그 시점의 상대 위치 |
| 2차 미분(APG) | a~e파 → b/a, c/a, d/a, e/a, aging index |
| 검출 | notch·APG 검출 성공 여부 (**소실 자체가 정보**) |

`CT/LVET` 은 심초음파의 **AT/ET**(acceleration time / ejection time)에 대응하며,
대동맥판막 협착 중증도 판별에서 AUC 0.88이 보고된 지표다.

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "src"))
from ppg_fm.config import load
from ppg_fm.morph.features import extract, beat_features, bandpass, segment_beats, FEATURE_NAMES

CFG  = load()
DATA = CFG["datasets"]["mimic_ext_ppg"]["root"]
INT  = os.path.join(ROOT, "data", "interim")
REP  = os.path.join(ROOT, "reports")

## 1. 한 세그먼트로 동작 확인

In [ ]:
import wfdb
idx = pd.read_parquet(os.path.join(INT, "seg_index.parquet"))
fp  = idx[idx.hq].iloc[0].folder_path

os.chdir(DATA)
rec = wfdb.rdrecord(fp)
sig = rec.p_signal[:, rec.sig_name.index("PLETH")]
print(f"{fp}  |  {rec.fs} Hz  |  {len(sig)} 샘플 ({len(sig)/rec.fs:.0f}초)")

feats, waves = extract(sig, rec.fs)
print(f"박동 {len(feats)}개, 정규화 파형 행렬 {waves.shape}")

## 2. 박동 분할 시각화

어디를 onset·peak 으로 잡는지 눈으로 확인한다.

In [ ]:
import matplotlib.pyplot as plt

x = bandpass(sig, rec.fs)
z = (x - x.mean()) / x.std()
bts = segment_beats(z, rec.fs)

t = np.arange(len(x)) / rec.fs
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.plot(t, x, lw=1.1, color="#52514e")
for on, sp, on2 in bts[:12]:
    ax.axvline(on/rec.fs, color="#2a78d6", lw=.8, alpha=.7)
    ax.plot(sp/rec.fs, x[sp], "o", ms=5, color="#eb6834")
ax.set_xlim(0, 8); ax.set_xlabel("time (s)"); ax.set_ylabel("PPG (a.u.)")
ax.set_title("박동 분할 — 파란 선 = onset, 주황 점 = systolic peak", loc="left")
for s in ("top","right"): ax.spines[s].set_visible(False)
plt.show()

## 3. 2차 미분(APG) 파 검출

APG의 a·b·c·d·e 파는 형태학 지표의 핵심이지만, **c·d 파는 동맥이 경직되면
생리적으로 융합되어 사라진다.**

중요한 설계 결정: 검출에 실패했을 때 **억지로 값을 만들지 않는다.**
단조 구간에서 argmax/argmin 을 취하면 물리적으로 존재하지 않는 파를
만들어내므로, 실제 국소극값(`find_peaks`)만 인정하고 없으면 `NaN` 을 둔다.
그리고 **그 결측률 자체를 특징으로 쓴다.**

In [ ]:
from scipy.signal import savgol_filter

on, sp, on2 = bts[3]
seg = x[on:on2]
s = (seg - seg.min()) / np.ptp(seg)
w = max(5, int(0.03*rec.fs) | 1)
d1 = savgol_filter(s, w, 3, deriv=1)
d2 = savgol_filter(s, w, 3, deriv=2)
tt = np.arange(len(s)) / rec.fs * 1000

fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for a_, y_, lab in zip(axes, [s, d1, d2], ["PPG (normalised)", "1st derivative", "2nd derivative (APG)"]):
    a_.plot(tt, y_, lw=1.6, color="#2a78d6"); a_.set_ylabel(lab, fontsize=9)
    a_.axhline(0, color="#c9c8c3", lw=.8)
    for sname in ("top","right"): a_.spines[sname].set_visible(False)
axes[-1].set_xlabel("time within beat (ms)")
axes[0].set_title("한 박동의 파형과 미분", loc="left")
plt.tight_layout(); plt.show()

f = beat_features(x, rec.fs, on, sp, on2)
for k in ["CT","LVET","CT_over_LVET","RI","IPA","b_over_a","c_over_a","d_over_a","e_over_a","aging_index"]:
    print(f"  {k:16s} {f[k]:8.3f}" if not np.isnan(f[k]) else f"  {k:16s}   (검출 실패)")

## 4. 추출 성공률

특징별로 얼마나 자주 계산되는지 — **낮은 추출률은 버그가 아니라 생리 현상일 수 있다.**

In [ ]:
rows = []
for fp_ in idx[idx.hq].sample(25, random_state=3).folder_path:
    try:
        r = wfdb.rdrecord(fp_)
        rows += extract(r.p_signal[:, r.sig_name.index("PLETH")], r.fs)[0]
    except Exception:
        pass

B = pd.DataFrame(rows)
print(f"박동 {len(B):,}개 (세그먼트 25개)\n")
print(f"{'feature':20s} {'추출률':>7s}  {'중앙값':>9s}  {'IQR'}")
for c in FEATURE_NAMES:
    v = B[c].dropna()
    if not len(v): continue
    print(f"{c:20s} {100*len(v)/len(B):6.1f}%  {v.median():9.3f}  [{v.quantile(.25):7.3f}, {v.quantile(.75):7.3f}]")

### 추출률 해석

| 특징 | 추출률 | 이유 |
|---|---|---|
| CT, W25/50/75, 기울기 | ~100% | 항상 정의됨 |
| LVET, IPA, b/a, e/a | ~99% | dicrotic notch(=APG e파)는 대부분 잡힘 |
| **RI, dT** | **~47%** | 이완기 피크는 동맥이 경직되면 **소실** |
| **c/a, d/a, aging_index** | **~36%** | c–d 파 쌍이 융합되어 사라짐 |

**문헌 부합**: 상용 장비 기반 연구에서도 c/a·d/a 의 재현도(ICC 0.53–0.72)가
b/a·e/a(0.80–0.87)보다 낮게 보고된다. 낮은 추출률은 구현 결함이 아니라
지표 자체의 성질이다.

### 값의 생리학적 타당성 확인
- `b/a` 음수 (a는 수축기 양의 정점, b는 그 직후 음의 골)
- `e/a` 양수 (이완기 초반 양의 파)
- `d/a` 음수 (c와 e 사이의 국소 최소)

→ 다음: `03_case_control_pilot.ipynb`